# FuGuard reproductio
1. **Runtime → Change runtime type → GPU** (T4 or better).
2. Run the cells below top to bottom. They unzip the project to `/content/FuGuard_Reproduction`, install the 2 missing packages, and smoke-test everything.
3. Open the numbered notebooks from the Colab file browser (left sidebar → `FuGuard_Reproduction/notebooks/...`) in the order listed in the table at the end.

All data, checkpoints and results are written to the Colab VM (`/content/FuGuard_Reproduction/{data,artifacts,results}`).
The VM is temporary: download results with `colab.zip_results()` (the 03b notebook does it too).

In [ ]:
import glob, os, zipfile
PROJECT = "/content/FuGuard_Reproduction"
if not os.path.exists(PROJECT):
    zips = sorted(glob.glob("/content/FuGuard_Reproduction*.zip"))
    if not zips:
        from google.colab import files
        files.upload()                      # pick FuGuard_Reproduction_Colab.zip
        zips = sorted(glob.glob("/content/FuGuard_Reproduction*.zip"))
    zipfile.ZipFile(zips[-1]).extractall("/content")
os.chdir(PROJECT)
print(os.getcwd(), os.listdir("."))

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > GPU, then re-run"

In [ ]:
import os, sys
os.environ["FUGUARD_ROOT"] = PROJECT
sys.path.insert(0, PROJECT)
from fuguard_repro import colab
from fuguard_repro.paths import ensure_dirs
ensure_dirs()
colab.status()

### Smoke test (synthetic data, no downloads) - confirms every method runs on this runtime

In [ ]:
from fuguard_repro.smoke import run_smoke_test
run_smoke_test()

### Recommended first real run (CIFAR-10, seed 0)
Prepare the data, then run FuGuard + FedSGA through the notebook's engine. This validates the whole pipeline before the big grids.

In [ ]:
from fuguard_repro.config import image_config
from fuguard_repro.data_image import prepare_image_data
prepare_image_data(image_config("CIFAR10"), 0)        # downloads CIFAR-10 to data/ (~170 MB) and builds the split

In [ ]:
from fuguard_repro.runner import run_grid, standard_jobs
run_grid("table2", ["CIFAR10"], standard_jobs(["fedsga", "fuguard"]), seeds=(0,), recovery_rounds=5)

### Long runs in the background (optional)
`colab.run_in_background` starts `scripts/run_experiment.py` as a separate process (survives a closed browser tab, not a recycled runtime).
Data must already be prepared (`scripts/prepare_data.py` or notebook 01a). Run one GPU job at a time.

In [ ]:
# p = colab.run_in_background(["table2", "--datasets", "CIFAR10", "--seeds", "0", "1"], log="logs/table2_cifar10.log")
# colab.tail_log("logs/table2_cifar10.log")

### Notebook order
| Step | Notebook (under `notebooks/`) |
|---|---|
| data | `01_data/01a_prepare_image_datasets` (+ `01b` AWE, `01c` data figures) |
| Table II / Fig. 6 | `02_experiments/02a_table2_protocol_5_recovery_rounds` |
| Table III | `02_experiments/02b_table3_protocol_50_recovery_rounds` |
| Table IV | `02_experiments/02c_ablation_table4` |
| Fig. 10 | `02_experiments/02d_sensitivity_fig10` |
| Fig. 9 | `02_experiments/02e_scalability_100_clients_fig9` |
| AWE Fig. 7/8 | `02_experiments/02f_awe_real_dataset_fig7_fig8` |
| tables + figures + download | `03_analysis/03a_tables`, `03b_figures` |

Save your work before the session ends:

In [ ]:
colab.zip_results()                 # downloads results + checkpoints as a zip
# colab.backup_to_drive()           # OPTIONAL: copy them to your Google Drive instead